<!-- dd:dd-lesson-eo-2 -->

# Reduce

*Einops · `eo-2`*

Work through this with the **Delta Drills** side panel open. It picks what you practise, sends you to the cell, and records how it went — you do not need to read this notebook in order.


In [ ]:
# === Delta Drills ===
# Which lesson this notebook is, for the side panel. Nothing to run.
DD_LESSON_ID = "eo-2"


In [ ]:
#@title 🔧 Delta Drills checker — run me first { display-mode: "form" }
# Delta Drills — problem checker. Generated; see scripts/colab_grader.py.
#
# `dd_check(<problem id>)` runs your `solve` against the same cases the tutor
# grades with, and tells you which ones failed. It reads `solve` out of the
# notebook, so define it (run your cell) before you check.
import base64
import json
import sys
import zlib

import numpy as np

# Filled in by the generated cell that follows this source: {qid: {fn, cases}}.
_DD_TESTS = {}

# Where the ARENA digits fixture is fetched from, also filled in by that cell.
_DD_FIXTURE_URL = ""
_DD_FIXTURE_PATH = "/delta_numbers.npy"

_DD_RTOL = 1e-5
_DD_ATOL = 1e-6


def _dd_install_fixtures():
    """Make `np.load('/delta_numbers.npy')` work here the way it does in the app.

    24 of the einops drills are written against the ARENA digits image, and the
    bank refers to it by an absolute path the backend rewrites at grade time
    (`code_runner.CODE_PREAMBLE`). Nothing rewrote it in a notebook, so those
    problems could not run at all in Colab — not the checker, not the starter
    code the learner was sent there to fill in. Downloaded on first use, so the
    six notebooks that never touch it never pay for it.
    """
    import os
    import urllib.request

    original = np.load
    if getattr(original, "_dd_patched", False):
        return

    def _load(file, *args, **kwargs):
        if str(file) == _DD_FIXTURE_PATH and not os.path.exists(_DD_FIXTURE_PATH):
            if not _DD_FIXTURE_URL:
                raise FileNotFoundError(
                    "This drill needs the ARENA digits fixture and no source was "
                    "compiled into this notebook — regenerate it."
                )
            urllib.request.urlretrieve(_DD_FIXTURE_URL, _DD_FIXTURE_PATH)
        return original(file, *args, **kwargs)

    _load._dd_patched = True
    np.load = _load


def _dd_load(blob):
    """The test payload, deflated and base64'd.

    Not encryption and not pretending to be — it is one `zlib.decompress` away.
    It is compressed because the payload for a 84-problem notebook is ~80 KB of
    JSON, and out of sight because an expanded grader cell would otherwise sit
    in the notebook spelling out the expected answer to every problem below it.
    """
    return json.loads(zlib.decompress(base64.b64decode(blob)).decode("utf-8"))


def _dd_preflight_torch():
    """Import torch once, here, where a failure can still be explained.

    Every drill cell opens with `import torch as t`, so the learner meets a
    broken torch install as a traceback through torch's own internals — the one
    reported was `AttributeError: partially initialized module 'torch' has no
    attribute 'fx'` from `torch/_export/utils.py`, raised while evaluating a
    function's annotations. That message names neither the cause nor the cure,
    and it is not even the real error: it is what a LATER import sees after an
    earlier one died partway and left the half-built module in `sys.modules`.
    Python does unwind a failed import normally, but a torch that was swapped
    on disk under a running kernel (a `pip install` mid-session) or shadowed by
    a stray `torch.py` gets far enough in to be cached before it falls over.

    So: purge the wreckage and retry ONCE, which is the whole fix whenever the
    first failure was transient, and report what actually broke when it is not.
    Importing torch in this cell rather than lazily is safe now in a way the
    `_dd_tensor` comment below still guards against for the per-comparison
    path — the bank is 448/448 torch and every notebook imports it a few cells
    down, so there is no numpy-only notebook left to charge for it.

    Never raises: a checker that refuses to load over this would take the
    lesson down with the runtime.
    """

    def _purge():
        # Submodules too, and that is the whole point. Python drops only the
        # module that raised, so `torch` goes and a `torch._export` imported
        # seconds earlier STAYS — and the next `import torch` re-runs
        # `torch/__init__.py` straight back into that stale submodule, which
        # reaches for a `torch.fx` the half-built parent has not bound yet.
        # Leaving one behind reproduces the bug instead of clearing it.
        for name in [n for n in sys.modules if n == "torch" or n.startswith("torch.")]:
            del sys.modules[name]

    def _usable(mod):
        # `import torch` does NOT re-execute a module already in sys.modules,
        # so a corpse left by a failed import is imported "successfully" and
        # the error surfaces later, from the learner's own cell. Judge the
        # object, not the statement: a torch that finished has both of these.
        return hasattr(mod, "fx") and hasattr(mod, "__version__")

    cached = sys.modules.get("torch")
    if cached is not None and not _usable(cached):
        _purge()

    for attempt in (1, 2):
        try:
            import torch
            if not _usable(torch):
                raise ImportError(
                    "torch imported but is only partially initialised "
                    "(no .fx) — an earlier import in this session died partway"
                )
            return True
        except Exception as exc:
            if attempt == 1:
                _purge()
                continue
            print(
                "⚠️  This runtime cannot import PyTorch, so no drill in this "
                "notebook will run.\n"
                "    %s: %s\n"
                "    Fix: Runtime ▸ Disconnect and delete runtime, then reopen "
                "this notebook and run\n"
                "    this cell first. If it comes back, check for a file named "
                "torch.py in /content,\n"
                "    and re-run any pip install BEFORE anything imports torch."
                % (type(exc).__name__, exc)
            )
    return False


def _dd_tensor(value):
    # torch only if something already imported it. numpy-only notebooks must
    # not pay a torch import to compare two lists of ints.
    torch = sys.modules.get("torch")
    return torch is not None and isinstance(value, torch.Tensor)


def _dd_close(a, b):
    """Tolerance compare, but ONLY when a float or complex is involved.

    torch defaults to float32 where numpy defaults to float64 and honest
    answers differ in reduction order, so exact equality fails correct work.
    Integer and boolean results stay exact — an index answer (argmax, nonzero,
    searchsorted) must never be fudged by a tolerance. Returns None to mean
    "not a float comparison, use exact equality".
    """
    try:
        floaty = any(
            np.issubdtype(x.dtype, np.floating) or np.issubdtype(x.dtype, np.complexfloating)
            for x in (a, b)
        )
        if not floaty:
            return None
        if a.shape != b.shape:
            return False
        return bool(np.allclose(a, b, rtol=_DD_RTOL, atol=_DD_ATOL, equal_nan=True))
    except Exception:
        return None


def _dd_array_equal(a, b):
    close = _dd_close(a, b)
    if close is not None:
        return close
    return bool(np.array_equal(a, b))


def _dd_equal(a, b):
    if _dd_tensor(a) or _dd_tensor(b):
        try:
            a2 = a.detach().cpu().numpy() if _dd_tensor(a) else np.asarray(a)
            b2 = b.detach().cpu().numpy() if _dd_tensor(b) else np.asarray(b)
            return _dd_array_equal(a2, b2)
        except Exception:
            # dtypes numpy cannot hold (bfloat16, conj views): equal tensors
            # must not grade as unequal — ask torch itself.
            torch = sys.modules.get("torch")
            if torch is not None and isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor):
                try:
                    return bool(torch.equal(a.detach().cpu().resolve_conj(),
                                            b.detach().cpu().resolve_conj()))
                except Exception:
                    return False
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        return _dd_array_equal(np.asarray(a), np.asarray(b))
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        if len(a) != len(b):
            return False
        return all(_dd_equal(x, y) for x, y in zip(a, b))
    close = _dd_close(np.asarray(a), np.asarray(b))
    if close is not None:
        return close
    return bool(a == b)


def _dd_seed():
    # The same seed before the actual-side and the expected-side setup runs, for
    # BOTH rngs: setup executes twice, so an unseeded torch.rand in a fixture
    # would hand the two sides different data and fail an honest answer.
    np.random.seed(0)
    torch = sys.modules.get("torch")
    if torch is not None:
        torch.manual_seed(0)


def _dd_show(value, limit=320):
    try:
        text = repr(value)
    except Exception as exc:
        text = "<unrepresentable: %s>" % type(exc).__name__
    text = " ".join(text.split()) if len(text) > limit else text
    if len(text) > limit:
        text = text[: limit - 1] + "…"
    return text


def dd_check(question_id, verbose=True):
    """Grade the `solve` you just defined against this problem's cases.

    Returns True when every case passes. Prints which ones did not, with the
    fixture, what was expected and what came back — a failing grade should be
    evidence you can act on, not a verdict.
    """
    qid = str(question_id)
    entry = _DD_TESTS.get(qid)
    if entry is None:
        print("No checker for problem %s in this notebook." % qid)
        return False

    # The learner's namespace, not this function's: `solve` lives in the cell
    # they ran, and in Colab that is the caller's globals.
    try:
        env = sys._getframe(1).f_globals
    except Exception:
        env = globals()

    fn_name = entry.get("fn") or "solve"
    if fn_name not in env:
        print("❌ `%s` is not defined yet — run your solution cell first." % fn_name)
        return False

    cases = entry.get("cases") or []
    failures = []
    for i, case in enumerate(cases, 1):
        # A fresh copy per case: fixtures are exec'd, and exec'ing them into the
        # notebook's own globals would quietly overwrite whatever the learner
        # named `x` two cells ago.
        ns = dict(env)
        try:
            if case.get("setup_code"):
                _dd_seed()
                exec(case["setup_code"], ns)
            actual = eval(case["call"], ns)
            expected_setup = case.get("expected_setup_code") or case.get("setup_code")
            if expected_setup:
                _dd_seed()
                exec(expected_setup, ns)
            expected = eval(case["expected_expr"], ns)
            if not _dd_equal(actual, expected):
                failures.append((i, case, _dd_show(expected), _dd_show(actual), ""))
        except Exception as exc:
            failures.append((i, case, "", "", "%s: %s" % (type(exc).__name__, exc)))

    total = len(cases)
    if not failures:
        print("✅ Problem %s — %d/%d cases passed." % (qid, total, total))
        return True

    print("❌ Problem %s — %d of %d cases failed." % (qid, len(failures), total))
    if verbose:
        for i, case, expected, actual, error in failures:
            print("\n  case %d" % i)
            if case.get("setup_code"):
                for line in case["setup_code"].strip().splitlines():
                    print("    given     %s" % line)
            print("    called    %s" % _dd_show_source(case.get("call", "")))
            if error:
                print("    raised    %s" % error)
            else:
                print("    expected  %s" % expected)
                print("    you got   %s" % actual)
    return False


def _dd_show_source(text, limit=160):
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

_DD_FIXTURE_URL = "https://raw.githubusercontent.com/AkiraTheSquid/arena-book-colab/main/ARENA_5.0/ch-1-foundations/numbers.npy"
_dd_preflight_torch()
_dd_install_fixtures()
_DD_TESTS = _dd_load(
    "eNrtW22P2jgQ/isRX5qcII1fk5xU6X7EfaOrii5puzoWEMlW3FX972fPjJ0QWAIJ2XbVIsiLPXE89jwz4/HwbSK4mvwZfJt8"
    "WpvTpNysvhaTaTC5X5RFaUrm3yZlUT1tP9xvloWleHjcbnZVUG1291+CRRlU79eL3S54F1TxZl2UYcingZgG5sijCFtareyD"
    "4UP5sC6rxfq+COE1oXkumprn/i7W5cZe1sXwZLHfFvdVsfxgLnbQxN+7p2IazFmcTANzuIsm36fBxT0Mq3ixW6w/F2Gup8Gy"
    "+ndbvKviT6vNohI8Ct4GMop3RfllsS1CZEGa71hMiDhLlWGD0Tm35ys5quL/it0GB53hN4qgcs7uTL2Ok5F6n9gp0FdPQRVX"
    "8KZwbj7ctiFNG+Yz3ihDF+9MJ42oZ0NF/eHxM3BBcpTFSS0ynKT+PCemhVOc2OLznOBwWX7MjbTXyg7dpeOPPa8hIJIuCCgE"
    "8mj8COSHqzuYJ/u6OFWeO2lrrmTP6SABWJDRaH13Gqg+XN/VGgnwOA1JNxYGSdBdAw36Vor/FBoYoqFbgHpDm2aBEKHsdUrj"
    "188qpLwLEmjbxIhWYW40K5gDOHIChIA7CUdFZRruUjhmtgwkKMenqRFGtIzjPbbDpCtWeI9NsRRaGaLRZzALMz6uTjfv4T0m"
    "uuGg4JdFI4smQ9GsVURTdRzW1LDU6SuHJYtBStXdq3bP5uSXKXDLTEFOfppwBQYvUMKBcoA0Cue5jcdK01A15O5Ky3XWh7Mv"
    "msHtbGSfDj2HmTiwZnk+3LeDouJhvdmWNbOPm6/F8uEx9Gyvt7GRyWX45u2yWFWLD+unx4/FrozX23/fRFFcbcJaamFmZywC"
    "q1+a9kyrB6Ni+rRqGPYyikHAzYMP68o0tNs8rZcH9Z9Wi8r0JGy+Scso+MPo8Sj4q8Z98yHTx2IVRi1YmceeG+rQQJCpxB3M"
    "kzxjmrNcsyxPU365s3MwqDQKvo/MdJvTTxyD/lhtiW7nqHzGOyo7lfacee8a5TkluCROoIcwXes3z/Fprm1pe1khxmad3CjG"
    "8cxrnzTBEpHT4GQwGPCUogHSGs+pck9lEkty127inQIErEz4zwHYA4giQufi7hxIOzDaB6J9EFrDU1gmcgNOlXEL0/5y2sSm"
    "cNis+/RkGM7aVni8FeKcy8y0LzOQSHcN8sWVfbPiUOOuocaMB0gcnIdg9giyLbjiYJgydbRAGHFEmFWFdECgni3wQ9IYldZl"
    "w4iKnxiTyevDJE8SkQrNec6z22DSieHzmOQjR21ayJP2WtZ4O8Zmm+JWiBTHboNHZNrWUWMiUtg1AB0OcAVj0bO2xqRMhmJy"
    "P3w1uD81Ovtur2IGi8FZEiN3MUQ2KFTxXN3lErK/ctWI2Lhg1TiAXVgSmrPmeBZ0zzDOadhFioQoEqJIHAVcGBnAciJL/KIT"
    "a6ldeg0tQGlQf+keoF/6ex5++DxcBWID2KfVKgxxDwv0EWxvjART2LyiZY2/vrrLjSiID0R3Rz0GqBYfK6yNQzqiceD1ZIzE"
    "Emz6zDgagIY5IBR7k2D9CNwfeuXmgSKG5oyASwlwqYekJgpNFJootKdQRKGIQhGF8hSSKCRRSKKQQIHK4XdPDnqCAidRoVEd"
    "kUqn7hTWUtv0Kh8e1lhLfaOualebYi3xRqymtbL+1d7czzzQtg3GI8WY5oHWzXWkq1U2yFS4Nsc3GO5NbbPB5Ujr/B4hchyj"
    "c3Hx/bnF/f7qpf2+Zzjc2AYQZGXDEyqTIuEJ14kSfeNt+3Yk3P6k/Z2PhLNRjRRYWolWl6EJ5tfZ3RNsHse+7U+fjH3Lo5Qa"
    "o1RGYpZBqgnH1BqPcQ33KbiGaSPczRgUMZbBiWu/p6uwxMbB7Ua6bOFNi58Jb68spmZwZwNqKjEPpTrPuWAGdzcKqDEKqOnz"
    "kBPdIjgghJSC1CTxjePV+sLtJT1qfMztKMH+EcJLN/aGmigZnJ63F8uDydWng10XLGnE8qRGMcWdOmWeNjSGQk1wub8AHDSi"
    "nJfkJKFBGJst5pOMBCYKIYtz5cspgehabn0Wa738h+xhqKeQKGW32qTWvCupdRCXea8JOw5gWAMNwZZkzDnBqAC5eg0oKXnj"
    "JCJGCacn4MTGy6KaJz5fhQLmpEuuc8Lb+UVCX5zox8dlrhGGgsiaOpUv1oyCWG+FduhjSBD0kKxTIYWLldSptM11zEu+8+rU"
    "Ig8jJ1uXxAFvIGEH6zt3PywXkkZmxsbOhnR9bmVYCHHrBN8je+pWIu/X/0yDj8zQ2jnrZHQaAPkz/LraLrZ9xq8VX5BFsrZ9"
    "1QLjWZdekHXe4QHP/IV4ZjH3ycE5XOc+RZhTBj3k/fKD9N8U7zFZmCu44y6RmGFDDFOHOdLy1Ft1JFcug5i7RGKM8XCfhMyQ"
    "kLl2OfaCUyozvoW7/nJsiWOPBdIK1i8F2adSstp3+CGzU1urq3noCMHXDLGXg1ji80T9H06cRRD0B6BG0j1tSaQ33ZI4p3Ps"
    "7HY4V4bTk96VLe/g3ukXEHevXfptPmh5gcfBX4qzphZp6o0m0mtkY/Kz0zWH2uVQI7R0wJVBUgQxa/x98AUGolce9IGAymP5"
    "dIC1YB2v8zU4/T/DEIXf/weN3pEC"
)
print("Delta Drills checker ready — 15 problems. Run dd_check(<problem number>) under any of them.")


<!-- dd:dd-kp-einops-reduce-model -->

## einops.reduce — dropping axes with an aggregation

`einops.reduce-model`


Where rearrange must keep every axis, **`einops.reduce` is allowed to drop
them — and you say HOW the dropped values collapse**:

> `einops.reduce(x, 'b c h w -> b', 'mean')`
> — c, h, w vanish from the pattern, so each output element is the mean
> over everything that vanished. The third argument names the aggregation:
> `'mean'`, `'max'`, `'min'`, `'sum'`, `'prod'`.

This is PyTorch's `dim=` reductions with the einsum-style deletion rule, in
einops clothing — three notations, one concept:

- `x.mean(dim=(1, 2, 3))` — axes by number.
- `t.einsum('bchw->b', x) / (c*h*w)` — sum by deletion, mean by hand.
- `reduce(x, 'b c h w -> b', 'mean')` — deletion by name, aggregation
  declared. (Note: unlike einsum, reduce does means/maxes natively — no
  divide-outside dance.)

Details that matter in the drills:

- **Partial drops**: `'b h w c -> b h w'` maxes only over channels;
  `'b c h w -> b c'` averages each channel map. Any subset of axes can go.
- **Keep a singleton**: writing `1` (or `()` — same thing) in the output
  where the dropped axis was — `'h w c -> 1 w c'` — keeps the result
  broadcast-ready against the input, einops' keepdim. This is exactly the
  `keepdim=True` story: reduce-then-broadcast pipelines (subtract each
  column's max…) want the singleton kept.
- **Reduce + rearrange compose**: the pattern can still permute the
  survivors while reducing (`'b c h w -> c b'` is legal), and — the next
  KP — parenthesized factors turn reduce into pooling.


Task: per-image scalar means; per-column max keeping a singleton row; and a
grayscale via mean-over-channels with the channel kept.


In [ ]:
import torch as t
import einops

x = t.arange(24.0).reshape(2, 3, 2, 2)      # (b, c, h, w)

# All of c, h, w collapse under 'mean' -> one number per image.
per_image = einops.reduce(x, 'b c h w -> b', 'mean')
assert per_image.tolist() == [5.5, 17.5]
assert t.allclose(per_image, x.mean(dim=(1, 2, 3)))   # the numpy twin

# Partial drop: mean over spatial only -> per-channel statistics.
per_channel = einops.reduce(x, 'b c h w -> b c', 'mean')
assert per_channel.shape == (2, 3)
assert per_channel[0, 0] == x[0, 0].mean()

# Keep-a-singleton: per-column max of an image, row axis kept as 1.
img = t.tensor([[1.0, 5.0],
                [7.0, 2.0]])
colmax = einops.reduce(img, 'h w -> 1 w', 'max')
assert colmax.shape == (1, 2)
assert colmax.tolist() == [[7.0, 5.0]]
# Why keep it: the (1, 2) result broadcasts straight back against (2, 2).
assert (img - colmax).shape == (2, 2)

# Grayscale keeping a trailing singleton channel: (b,h,w,c) -> (b,h,w,1).
imgs = t.ones((2, 2, 2, 3))
gray = einops.reduce(imgs, 'b h w c -> b h w 1', 'mean')
assert gray.shape == (2, 2, 2, 1)
print("'b c h w -> b'  ", per_image, " (one number per image)")
print("'b c h w -> b c'", tuple(per_channel.shape), per_channel[0])
print("'h w -> 1 w'    ", colmax, "shape", tuple(colmax.shape),
      "-> broadcasts back to", tuple((img - colmax).shape))
print("grayscale keeps the channel axis:", tuple(imgs.shape), "->",
      tuple(gray.shape))




Why each step:

1. The `dim=` twin assert carries your existing axis intuition into the new
   notation: names deleted ↔ axis numbers listed. After a few reps the
   named form usually reads faster — especially at rank 4+.
2. In the singleton example, the follow-up subtraction is the POINT: the
   kept `1` is what makes reduce-then-operate pipelines shape-safe, same as
   keepdims in np-3.
3. The grayscale line matches a drill's exact contract ('-> b h w 1');
   note reduce handles mean natively — resist the einsum habit of dividing
   afterwards.


<!-- dd:dd-q325 -->

### Problem 325 · faded — your turn

Each image of a channels-first batch → one scalar mean.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1., 1.])
```


In [ ]:
import torch as t
import einops

def solve(arr):
    """(b, c, h, w) -> (b,): mean over channels and pixels."""
    return einops.reduce(arr, '_____', 'mean')


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((2, 3, 2, 2))))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(325)


In [ ]:
#@title 💡 Solution — Problem 325
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    return einops.reduce(arr, 'b c h w -> b', 'mean')


print(solve(t.ones((2, 3, 2, 2))))


<!-- dd:dd-q328 -->

### Problem 328 · guided

Write a function solve(img) that takes a channels-last image of shape (h, w, c) and returns the (w, c) tensor averaging over the HEIGHT axis: each (column, channel) pair's mean down the image.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[2., 3.],
        [4., 5.]])
```


<details>
<summary>Hints</summary>

1. (h, w, c) image, average over the HEIGHT axis only → (w, c).
2. One name disappears from the pattern; the aggregation string says how.
3. `einops.reduce(img, 'h w c -> w c', 'mean')`.

</details>


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img):
    """Return the (w, c) array averaging over the HEIGHT axis."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(328)


In [ ]:
#@title 💡 Solution — Problem 328
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img):
    return einops.reduce(img, 'h w c -> w c', 'mean')


print(solve(t.arange(8.0).reshape(2, 2, 2)))


<!-- dd:dd-q326 -->

### Problem 326 · independent

Write a function solve(arr) that takes a channels-last batch of shape (b, h, w, c) and returns the (b, h, w) tensor where each output pixel is the MAXIMUM across that pixel's channels.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[1., 3.],
         [5., 7.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return the (b, h, w) array where each output pixel is the MAXIMUM across that pixel's channels."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 2, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(326)


In [ ]:
#@title 💡 Solution — Problem 326
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    return einops.reduce(arr, 'b h w c -> b h w', 'max')


print(solve(t.arange(8.0).reshape(1, 2, 2, 2)))


<!-- dd:dd-q367 -->

### Problem 367 · independent

Write a function solve(arr) that takes a channels-first batch (b, c, h, w) and returns the (b, c) tensor of PER-CHANNEL spatial means: each value averages one channel's h x w map.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1.5000, 5.5000]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return the (b, c) array of PER-CHANNEL spatial means."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 2, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(367)


In [ ]:
#@title 💡 Solution — Problem 367
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    return einops.reduce(arr, 'b c h w -> b c', 'mean')


print(solve(t.arange(8.0).reshape(1, 2, 2, 2)))


<!-- dd:dd-q399 -->

### Problem 399 · independent

Write solve(imgs) for a float (B, H, W, C) batch: grayscale each image by AVERAGING across the color channels, keeping a trailing singleton channel — output (B, H, W, 1). Pattern: einops.reduce 'b h w c -> b h w ()' with 'mean'.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[[205.],
          [205.],
          [205.],
          ...,
          [205.],
          [205.],
          [205.]],

         [[205.],
          [205.],
          [205.],
          ...,
          [205.],
          [205.],
          [205.]],

         [[205.],
          [205.],
          [205.],
          ...,
          [205.],
          [205.],
          [205.]],
… (truncated)
```


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')).to(t.float32), 1, -1)
imgs = arr

def solve(imgs):
    # Write your solution here
    return None

print(solve(imgs))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(399)


In [ ]:
#@title 💡 Solution — Problem 399
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')).to(t.float32), 1, -1)
imgs = arr

def solve(imgs):
    return einops.reduce(imgs, 'b h w c -> b h w ()', 'mean')

print(solve(imgs))


<!-- dd:dd-q402 -->

### Problem 402 · independent

Write solve(img) for an (H, W, C) image: compute each COLUMN's per-channel maximum (reduce the height axis to a singleton: 'h w c -> () w c' with 'max') and subtract it from the image. Return img - mx. (uint8 wraparound on underflow is expected — same on both sides.)

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],

        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],

        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
… (truncated)
```


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img = arr[3]

def solve(img):
    # Write your solution here
    return None

print(solve(img))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(402)


In [ ]:
#@title 💡 Solution — Problem 402
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img = arr[3]

def solve(img):
    mx = einops.reduce(img, 'h w c -> () w c', 'max')
    return img - mx

print(solve(img))


<!-- dd:dd-q332 -->

### Problem 332 · independent

Write solve(img) for an (H, W, C) image: compute the maximum over each ROW — reducing width and channels to singletons, 'h w c -> h () c' is close but the row max is over BOTH w and c per h and c... precisely: reduce 'h w c -> h () c' with 'max' takes each row's per-channel max over width; then subtract that from the image (broadcasting back). Return img - mx. (uint8 wraparound on underflow is expected — same on both sides.)

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],

        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],

        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
… (truncated)
```


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img = arr[0]

def solve(img):
    # Write your solution here
    return None

print(solve(img))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(332)


In [ ]:
#@title 💡 Solution — Problem 332
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img = arr[0]

def solve(img):
    mx = einops.reduce(img, 'h w c -> h () c', 'max')
    return img - mx

print(solve(img))


<!-- dd:dd-q340 -->

### Problem 340 · independent

Write a function solve(x) that takes a 4-D tensor of shape (b, c, h, w) and CENTERS each (batch, channel) feature map: subtract from every pixel the mean over ITS OWN map's h and w. Return the same shape; the trick is reducing to 'b c 1 1' so the means broadcast back.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[-1.5000, -0.5000],
          [ 0.5000,  1.5000]],

         [[-1.5000, -0.5000],
          [ 0.5000,  1.5000]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x):
    """Implement the described contraction and return the result."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 2, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(340)


In [ ]:
#@title 💡 Solution — Problem 340
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x):
    return x - einops.reduce(x, 'b c h w -> b c 1 1', 'mean')


print(solve(t.arange(8.0).reshape(1, 2, 2, 2)))


<!-- dd:dd-q370 -->

### Problem 370 · independent

Write a function solve(x) that takes a batch (b, c, h, w) and centers each CHANNEL across the WHOLE BATCH and all pixels: subtract from every value its channel's mean over (b, h, w). Reduce to '1 c 1 1' so the means broadcast — BatchNorm-style centering (contrast per-image centering, which keeps b).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[-3.5000, -2.5000],
          [-1.5000, -0.5000]]],


        [[[ 0.5000,  1.5000],
          [ 2.5000,  3.5000]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x):
    """Implement the described contraction and return the result."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(370)


In [ ]:
#@title 💡 Solution — Problem 370
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x):
    return x - einops.reduce(x, 'b c h w -> 1 c 1 1', 'mean')


print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


#### Common mistakes

- **"reduce is rearrange with a mode argument."** — rearrange forbids
  dropping names; reduce requires it (something must reduce). They share
  the pattern grammar, not the contract.
- **"Means still need dividing outside, like einsum."** — reduce's third
  argument does real means/maxes/mins natively. The divide-outside habit
  is einsum-specific.
- **"'h w -> w' and 'h w -> 1 w' are the same reduction."** — Same numbers,
  different SHAPE: (w,) vs (1, w). The singleton version survives
  broadcasting against the original; the bare version may misalign
  (np-3's keepdims lesson, verbatim). Drills specify which they grade.


<!-- dd:dd-kp-einops-pooling -->

## Pooling with factored axes

`einops.pooling`


Combine reduce's aggregation with split's parentheses and you get POOLING —
window-wise downsampling — as pure notation:

> `einops.reduce(x, 'b c (h h2) (w w2) -> b c h w', 'mean', h2=2, w2=2)`

Read the input side as a split: height factors into (h blocks × h2 rows),
width into (w × w2). The output keeps the block coordinates (h, w) and
DROPS the within-window names (h2, w2) — so each output pixel aggregates
its h2×w2 window. That's non-overlapping average pooling; `'max'` makes it
max pooling. The mental model:

> **split the axis into (keep × window), reduce away the window.**

Variants the drills exercise:

- **Any window size / rectangle**: `h2=3, w2=3` for 3×3; the factors need
  not match.
- **Any rank**: a 5-D volume pools with three factored axes —
  `'b c (x a) (y b2) (z c2) -> b c x y z'` — nothing new, one more group.
- **Pool one axis only**: halve the width by averaging adjacent column
  pairs: `'b h (w w2) c -> b h w c', w2=2` — "adjacent pairs" is a
  length-2 window on that axis alone. Temporal downsampling
  ('b c (t two) -> b c t') is the same idea on sequences.
- **Pooling + flatten, etc.**: since it's all one pattern language, pooling
  composes freely with merges in the same call.

One requirement: non-overlapping windows must TILE the axis — sizes must
divide exactly (drills guarantee it; real code pads first). Overlapping /
strided pooling is outside reduce's power — that's `x.unfold(...)`
and the pooling layers' territory.


Task: 2×2 average pooling on a batch; then halving width by averaging
adjacent column pairs.


In [ ]:
import torch as t
import einops

x = t.arange(16.0).reshape(1, 1, 4, 4)      # (b, c, H, W)

# Split H into (2 blocks x 2 rows), W likewise; reduce the window names.
pooled = einops.reduce(x, 'b c (h h2) (w w2) -> b c h w', 'mean', h2=2, w2=2)
assert pooled.shape == (1, 1, 2, 2)
# Window (0,0) = mean of [[0,1],[4,5]] = 2.5:
assert pooled[0, 0].tolist() == [[2.5, 4.5],
                                 [10.5, 12.5]]

# Max pooling is the same pattern, different aggregation.
mx = einops.reduce(x, 'b c (h h2) (w w2) -> b c h w', 'max', h2=2, w2=2)
assert mx[0, 0].tolist() == [[5.0, 7.0],
                             [13.0, 15.0]]

# One-axis pooling: average adjacent COLUMN pairs, everything else intact.
imgs = t.arange(8.0).reshape(1, 2, 4, 1)    # (b, h, w=4, c)
halved = einops.reduce(imgs, 'b h (w w2) c -> b h w c', 'mean', w2=2)
assert halved.shape == (1, 2, 2, 1)
assert halved[0, 0, :, 0].tolist() == [0.5, 2.5]   # (0+1)/2, (2+3)/2
print(x[0, 0], "\n")
print("mean-pooled 2x2 ->\n", pooled[0, 0])
print("max-pooled  2x2 ->\n", mx[0, 0])
print("column pairs averaged:", imgs[0, 0, :, 0].tolist(), "->",
      halved[0, 0, :, 0].tolist())




Why each step:

1. Hand-verify ONE window ([[0,1],[4,5]] → 2.5) and trust the pattern for
   the rest — the same one-element discipline as every layout KP, now with
   an aggregation attached.
2. mean→max changing only the string argument shows where the operation
   lives: geometry in the pattern, semantics in the aggregation. Swap
   either independently.
3. In the one-axis case, note which name went INSIDE the parens: the axis
   being pooled. Everything outside parens rides along — that's how the
   pattern scales to 5-D volumes without new ideas.


<!-- dd:dd-q324 -->

### Problem 324 · faded — your turn

2×2 non-overlapping average pooling on (B, C, H, W).

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[[255., 255., 255.,  ..., 255., 255., 255.],
          [255., 255., 255.,  ..., 255., 255., 255.],
          [255., 255., 255.,  ..., 255., 255., 255.],
          ...,
          [255., 255., 255.,  ..., 255., 255., 255.],
          [255., 255., 255.,  ..., 255., 255., 255.],
          [255., 255., 255.,  ..., 255., 255., 255.]],

         [[205., 205., 205.,  ..., 205., 205., 205.],
          [205., 205., 205.,  ..., 205., 205., 205.],
          [205., 205., 205.,  ..., 205., 205., 205.],
          ...,
          [205., 205., 205.,  ..., 205., 205., 205.],
          [205., 205., 205.,  ..., 205., 205., 205.],
          [205., 205., 205.,  ..., 205., 205., 205.]],

         [[155., 155., 155.,  ..., 155., 155., 155.],
          [155., 155., 155.,  ..., 155., 155., 155.],
          [155., 155., 155.,  ..., 155., 155., 155.],
          ...,
          [155., 155., 155.,  ..., 155., 155., 155.],
          [155., 155., 155.,  ..., 155., 155., 155.],
          [155., 155., 155.,  ..., 155., 155., 155.]]],
… (truncated)
```


In [ ]:
import torch as t
import einops

def solve(x):
    """Halve H and W by averaging each 2x2 window."""
    return einops.reduce(x, '_____', 'mean', h2=2, w2=2)


arr = t.tensor(np.load('/delta_numbers.npy')).to(t.float32)
x = arr


print(solve(x))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(324)


In [ ]:
#@title 💡 Solution — Problem 324
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.tensor(np.load('/delta_numbers.npy')).to(t.float32)
x = arr

def solve(x):
    return einops.reduce(x, 'b c (h h2) (w w2) -> b c h w', 'mean', h2=2, w2=2)

print(solve(x))


<!-- dd:dd-q363 -->

### Problem 363 · guided

Write solve(img) for a float (C, H, W) image with H and W divisible by 3: 3×3 non-overlapping average pooling — H and W shrink by 3×, channels unchanged. einops.reduce with 'mean' and named window factors.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[255., 255., 255.,  ..., 255., 255., 255.],
         [255., 255., 255.,  ..., 255., 255., 255.],
         [255., 255., 255.,  ..., 255., 255., 255.],
         ...,
         [255., 255., 255.,  ..., 255., 255., 255.],
         [255., 255., 255.,  ..., 255., 255., 255.],
         [255., 255., 255.,  ..., 255., 255., 255.]],

        [[205., 205., 205.,  ..., 205., 205., 205.],
         [205., 205., 205.,  ..., 205., 205., 205.],
         [205., 205., 205.,  ..., 205., 205., 205.],
         ...,
         [205., 205., 205.,  ..., 205., 205., 205.],
         [205., 205., 205.,  ..., 205., 205., 205.],
         [205., 205., 205.,  ..., 205., 205., 205.]],

        [[155., 155., 155.,  ..., 155., 155., 155.],
         [155., 155., 155.,  ..., 155., 155., 155.],
         [155., 155., 155.,  ..., 155., 155., 155.],
         ...,
         [155., 155., 155.,  ..., 155., 155., 155.],
         [155., 155., 155.,  ..., 155., 155., 155.],
         [155., 155., 155.,  ..., 155., 155., 155.]]])
```


<details>
<summary>Hints</summary>

1. 3×3 average pooling on a channels-first single image (c, h, w) — same
   split-and-reduce, no batch axis.
2. Window factors are 3 this time; both spatial axes factor.
3. `'c (h h3) (w w3) -> c h w', h3=3, w3=3` with 'mean'.

</details>


In [ ]:
import torch as t
import einops

arr = t.tensor(np.load('/delta_numbers.npy')).to(t.float32)
img = arr[0]

def solve(img):
    # Write your solution here
    return None

print(solve(img))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(363)


In [ ]:
#@title 💡 Solution — Problem 363
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.tensor(np.load('/delta_numbers.npy')).to(t.float32)
img = arr[0]

def solve(img):
    return einops.reduce(img, 'c (h h2) (w w2) -> c h w', 'mean', h2=3, w2=3)

print(solve(img))


<!-- dd:dd-q368 -->

### Problem 368 · independent

Write a function solve(x3d) that takes a 5-D volume batch (b, c, x, y, z) with all three spatial dims EVEN and max-pools it with non-overlapping 2x2x2 windows: return shape (b, c, x//2, y//2, z//2) via 'b c (x 2) (y 2) (z 2) -> b c x y z' with 'max'. (Note the anonymous literal 2s in the pattern.)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[[ 7.]]],


         [[[15.]]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x3d):
    """Return shape (b, c, x//2, y//2, z//2) via 'b c (x 2) (y 2) (z 2) -> b c x y z' with 'max'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(16.0).reshape(1, 2, 2, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(368)


In [ ]:
#@title 💡 Solution — Problem 368
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x3d):
    return einops.reduce(x3d, 'b c (x two1) (y two2) (z two3) -> b c x y z', 'max', two1=2, two2=2, two3=2)


print(solve(t.arange(16.0).reshape(1, 2, 2, 2, 2)))


<!-- dd:dd-q354 -->

### Problem 354 · independent

Write a function solve(arr) that takes a channels-last batch (b, h, w, c) and subtracts each (batch, channel) pair's SPATIAL MINIMUM from its own map: reduce with 'b h w c -> b () () c' and 'min', then subtract with broadcasting. Every map's minimum becomes exactly 0.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[0.],
          [1.]],

         [[2.],
          [3.]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Implement the described contraction and return the result."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(4.0).reshape(1, 2, 2, 1)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(354)


In [ ]:
#@title 💡 Solution — Problem 354
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    return arr - einops.reduce(arr, 'b h w c -> b () () c', 'min')


print(solve(t.arange(4.0).reshape(1, 2, 2, 1)))


<!-- dd:dd-q336 -->

### Problem 336 · independent

Write a function solve(arr, k, b1) that takes a channels-first batch (b, c, h, w) — h, w divisible by k, b divisible by b1 — and does two things at once: max-pool each image with non-overlapping k x k windows AND tile the batch into a b1-row grid. Return shape (c, b1*(h//k), (b//b1)*(w//k)) via einops.reduce with pattern '(b1 b2) c (h h2) (w w2) -> c (b1 h) (b2 w)' and 'max'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[ 5.,  7.],
         [13., 15.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, k, b1):
    """Implement the described contraction and return the result."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(16.0).reshape(1, 1, 4, 4), 2, 1))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(336)


In [ ]:
#@title 💡 Solution — Problem 336
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, k, b1):
    return einops.reduce(arr, '(b1 b2) c (h h2) (w w2) -> c (b1 h) (b2 w)', 'max', h2=k, w2=k, b1=b1)


print(solve(t.arange(16.0).reshape(1, 1, 4, 4), 2, 1))


<!-- dd:dd-q377 -->

### Problem 377 · independent

Write a function solve(x, k) that takes a batch (b, c, h, w) — h, w divisible by k — and does max-pooling with k x k windows FOLLOWED by a full flatten: return shape (b, c*(h//k)*(w//k)) in ONE einops.reduce: 'b c (h k1) (w k2) -> b (c h w)' with 'max'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ 5.,  7., 13., 15.]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x, k):
    """Return shape (b, c*(h//k)*(w//k)) in ONE einops.reduce."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(16.0).reshape(1, 1, 4, 4), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(377)


In [ ]:
#@title 💡 Solution — Problem 377
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x, k):
    return einops.reduce(x, 'b c (h k1) (w k2) -> b (c h w)', 'max', k1=k, k2=k)


print(solve(t.arange(16.0).reshape(1, 1, 4, 4), 2))


#### Common mistakes

- **"Pooling needs a framework (torch.nn.AvgPool2d) or a loop."** — Non-
  overlapping pooling is reshape+reduce, which is exactly what the factored
  pattern states. Frameworks add padding/stride options; the core is this.
- **"The window names (h2, w2) must be called that."** — Any names; what
  matters is they appear inside input parens and NOT in the output. The
  kept block-count names are the ones that survive.
- **"reduce can do stride-1 (overlapping) pooling too."** — No: factored
  axes tile the input disjointly. Overlap = sliding_window_view + reduction
  on `x.unfold(...)`. "Non-overlapping" in a task is your green light for
  the einops form.
